# 02. Latent Dirichlet Allocation (LDA)

**Goal:** Train an LDA model to discover 10 hidden topics in the news headlines using raw counts (Bag of Words).

In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

## 1. Load Data

In [2]:
df = pd.read_csv('../data/processed/processed_headlines.csv')
df = df.dropna(subset=['lemmatized'])

> **📌 Decision Note — Why Vectorization for LDA?**
>
> **Chosen approach:** CountVectorizer (Bag of Words)
>
> **Why this works:** LDA is a probabilistic graphical model that assumes documents are generated by drawing topics, and then drawing words from those topics based on counts.
>
> **Alternatives we could have used:**
> | Option | Pros | Cons |
> |--------|------|------|
> | TF-IDF Vectorizer | Downweights common words natively | LDA's statistical assumptions are fundamentally based on discrete integer counts, not continuous TF-IDF scores. |
> | Word Embeddings | Captures deep semantic meaning | LDA cannot process dense embeddings. Requires entirely different architectures (like Top2Vec or BERTopic). |
>
> **Why we chose this over alternatives:** Mathematical compatibility dictates that standard LDA must use raw word counts.

## 2. Vectorization

In [3]:
tf_vectorizer = CountVectorizer(max_df=0.95, min_df=2, max_features=5000)
tf = tf_vectorizer.fit_transform(df['lemmatized'])
tf_feature_names = tf_vectorizer.get_feature_names_out()

## 3. Fit LDA Model

In [4]:
lda = LatentDirichletAllocation(n_components=10, random_state=42, learning_method='online')
lda.fit(tf)

,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'online'
,"random_state random_state: int, RandomState instance or None, default=NonePass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",10
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0


## 4. Extract and Display Topics

In [5]:
def display_topics(model, feature_names, no_top_words):
    for topic_idx, topic in enumerate(model.components_):
        print(f'Topic {topic_idx}:')
        print(' '.join([feature_names[i] for i in topic.argsort()[:-no_top_words - 1:-1]]))

display_topics(lda, tf_feature_names, 10)

Topic 0:
water face claim first open north boost final abc sex
Topic 1:
council court hospital change case worker accused talk ban green
Topic 2:
win say nsw plan government missing minister national queensland power
Topic 3:
election two may group coronavirus one centre rate fight community
Topic 4:
police man death year hit murder charged crash south cut
Topic 5:
australia fire get child farmer rise trial road say could
Topic 6:
new interview car set found fear test dy record driver
Topic 7:
govt qld health school market job price urged labor family
Topic 8:
woman sydney home world coast new drug service china cup
Topic 9:
australian back call report take day attack help killed rural


In [6]:
joblib.dump(lda, '../models/lda_model.pkl')

['../models/lda_model.pkl']